In [1]:
import pandas as pd

ratings = pd.read_csv(
    "data/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
)

In [2]:
ratings.shape

(1000209, 4)

In [3]:
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
ratings["user_id"].nunique()

6040

In [5]:
ratings["movie_id"].nunique()

3706

In [6]:
ratings["date"] = pd.to_datetime(ratings["timestamp"],unit="s")

In [7]:
ratings.head()

,user_id,movie_id,rating,timestamp,date
0,1,1193,5,978300760,2000-12-31 22:12:40
1,1,661,3,978302109,2000-12-31 22:35:09
2,1,914,3,978301968,2000-12-31 22:32:48
3,1,3408,4,978300275,2000-12-31 22:04:35
4,1,2355,5,978824291,2001-01-06 23:38:11


In [8]:
ratings["date"].min() , ratings["date"].max()

(Timestamp('2000-04-25 23:05:32'), Timestamp('2003-02-28 17:49:50'))

In [9]:
ratings["date"].dt.to_period("M").value_counts().sort_index()

date
2000-04     11396
2000-05     67437
2000-06     54486
2000-07     90334
2000-08    182109
2000-09     52421
2000-10     42294
2000-11    290793
2000-12    113487
2001-01     18004
2001-02      8136
2001-03      6083
2001-04      5171
2001-05      4939
2001-06      4981
2001-07      4765
2001-08      4454
2001-09      3077
2001-10      2192
2001-11      2760
2001-12      3496
2002-01      3216
2002-02      2496
2002-03      2454
2002-04      2840
2002-05      1902
2002-06      1643
2002-07      1905
2002-08      2111
2002-09      1293
2002-10      1014
2002-11      1908
2002-12      1264
2003-01      1852
2003-02      1496
Freq: M, Name: count, dtype: int64

In [10]:
before_2001 = (ratings["date"] < "2001-01-01").sum()
before_2001, before_2001 / len(ratings)

(np.int64(904757), np.float64(0.9045679452994324))

In [11]:
pd.to_datetime(ratings["timestamp"].quantile(0.8), unit="s")

Timestamp('2000-12-02 14:52:18')

In [12]:
cutoff = pd.to_datetime(ratings["timestamp"].quantile(0.8), unit="s")

train = ratings[ratings["date"] < cutoff]
test = ratings[ratings["date"] >= cutoff]

In [13]:
len(train), len(test), len(train) + len(test)

(800164, 200045, 1000209)

In [14]:
popularity = train["movie_id"].value_counts()

In [15]:
popularity.head(10)

movie_id
2858    2902
260     2516
1196    2516
1210    2457
589     2284
2028    2245
480     2234
1270    2184
2571    2172
1580    2157
Name: count, dtype: int64

In [16]:
top_10 = popularity.head(10).index.tolist()
top_10

[2858, 260, 1196, 1210, 589, 2028, 480, 1270, 2571, 1580]

In [17]:
movies = pd.read_csv(
    "data/movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1",
)

In [18]:
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [19]:
movies[movies["movie_id"].isin(top_10)]

,movie_id,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
476,480,Jurassic Park (1993),Action|Adventure|Sci-Fi
585,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi|Thriller
1178,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1192,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
1250,1270,Back to the Future (1985),Comedy|Sci-Fi
1539,1580,Men in Black (1997),Action|Adventure|Comedy|Sci-Fi
1959,2028,Saving Private Ryan (1998),Action|Drama|War
2502,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
2789,2858,American Beauty (1999),Comedy|Drama


In [20]:
test_likes = test[test["rating"] >= 4]

In [21]:
len(test_likes)

112264

In [22]:
liked_by_user = test_likes.groupby("user_id")["movie_id"].apply(set)

In [23]:
liked_by_user

user_id
1       {1, 2692, 260, 1028, 1287, 1029, 1545, 1035, 5...
2       {1537, 515, 3334, 648, 1544, 265, 2571, 3468, ...
3       {260, 1291, 653, 1304, 1049, 2081, 2470, 552, ...
4       {480, 2366, 1954, 2947, 260, 2692, 2951, 1097,...
5       {2560, 515, 3083, 2571, 2580, 1046, 29, 32, 34...
                              ...                        
6001    {481, 965, 3751, 2600, 457, 2346, 3947, 3147, ...
6002    {2946, 2819, 2947, 2948, 2949, 1927, 1419, 909...
6016    {930, 3685, 1639, 3245, 339, 3894, 3129, 3834,...
6028                                               {3000}
6040    {1921, 1673, 2571, 2575, 2068, 1947, 3362, 272...
Name: movie_id, Length: 1762, dtype: object

In [24]:
len(liked_by_user)

1762

In [25]:
top_10_set = set(top_10)

def precision_at_10(liked_set):
    hits = len(top_10_set & liked_set)
    return hits / 10

In [26]:
scores = liked_by_user.apply(precision_at_10)
scores.mean()

np.float64(0.19324631101021567)